# 02 — Limpieza y Saneamiento (Entrega 2)

Objetivo: transformar datos crudos a dataset analítico listo para Tableau.

Operaciones:
1. Cargar y validar integridad
2. Deduplicación de eventos
3. Tratamiento de valores nulos
4. Corrección de tipos de datos
5. Validación y filtrado de outliers
6. Homologación de categorías
7. JOIN de tablas
8. Generación de bitácora
9. Exportación a `data/interim/inventory_v1.csv`

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
import os

# Crear directorio interim si no existe
os.makedirs('../data/interim', exist_ok=True)

print('Librerías importadas correctamente.')

# Cargar datos raw
catalog = pd.read_csv('../data/raw/catalog_raw.csv')
movements = pd.read_csv('../data/raw/movements_raw.csv')

# Convertir timestamp a datetime
movements['timestamp'] = pd.to_datetime(movements['timestamp'])

# Inicializar bitácora
log = {
    'timestamp_ejecucion': datetime.now().isoformat(),
    'registros_iniciales': {
        'catalog': len(catalog),
        'movements': len(movements)
    },
    'transformaciones': {}
}

print(f'Catálogo:    {catalog.shape[0]} filas × {catalog.shape[1]} columnas')
print(f'Movimientos: {movements.shape[0]} filas × {movements.shape[1]} columnas')
print(f'\nColumnas catálogo: {list(catalog.columns)}')
print(f'Columnas movimientos: {list(movements.columns)}')

In [ ]:
# Detectar y remover duplicados de event_id en movements
duplicados_detectados = movements['event_id'].duplicated().sum()
print(f'Duplicados detectados (event_id): {duplicados_detectados}')

# Remover duplicados, mantener primer registro
movements = movements.drop_duplicates(subset=['event_id'], keep='first')

log['transformaciones']['deduplicacion'] = {
    'duplicados_removidos': duplicados_detectados,
    'registros_despues': len(movements)
}

print(f'Movimientos después de deduplicación: {len(movements)} filas')

## Paso 2: Deduplicación

In [ ]:
# Detectar outliers en calorías (umbral 900 kcal)
outliers_cal = catalog[catalog['calories_100g'] > 900]
print(f'Outliers detectados en calories_100g (>900): {len(outliers_cal)}')
if len(outliers_cal) > 0:
    print(outliers_cal[['product_id', 'product_name', 'calories_100g']])

# Remover outliers
registros_antes = len(catalog)
catalog = catalog[catalog['calories_100g'] <= 900]
outliers_removidos = registros_antes - len(catalog)

print(f'\nRegistros removidos: {outliers_removidos}')
print(f'Registros en catálogo después: {len(catalog)}')

log['transformaciones']['validacion_outliers'] = {
    'outliers_detectados': len(outliers_cal),
    'outliers_removidos': outliers_removidos,
    'umbral_calories_100g': 900,
    'registros_catalog_despues': len(catalog)
}

# Mapeo de categorías basado en el análisis de datos reales
# Las categorías originales son multiidioma y de granularidad inconsistente
category_mapping = {
    'Alimentos y bebidas de origen vegetal': 'Plant-based Foods',
    'Aliments d\'origine végétale': 'Plant-based Foods',
    'Aliments et boissons à base de végétaux': 'Plant-based Foods',
    'Bevande e preparati per bevande': 'Beverages',
    'Beverages and beverages preparations': 'Beverages',
    'Boissons et préparations de boissons': 'Beverages',
    'Botanas': 'Snacks',
    'Snack': 'Snacks',
    'Snacks': 'Snacks',
    'Breakfasts': 'Breakfast',
    'Petit-déjeuners': 'Breakfast',
    'en:breakfasts': 'Breakfast',
    'Condiments': 'Condiments',
    'Red pestos': 'Condiments',
    'Koek': 'Snacks',
    'Lanches comida': 'Snacks',
    'Pâtes à tartiner au chocolat': 'Spreads',
    'Produits laitiers': 'Dairy',
    'produits-laitiers': 'Dairy',
    'Pflanzliche Lebensmittel und Getränke': 'Plant-based Foods',
    'Plant-based foods and beverages': 'Plant-based Foods',
    'en:haverdrink': 'Beverages'
}

# Aplicar mapeo
catalog['category_name'] = catalog['category'].map(category_mapping).fillna('Other')

print('Distribución de categorías:')
print(catalog['category_name'].value_counts())

log['transformaciones']['homologacion_categorias'] = {
    'mapeo_aplicado': True,
    'categorias_distintas': catalog['category_name'].nunique()
}

In [ ]:
nulos_iniciales = catalog.isnull().sum()
print('Nulos iniciales en catálogo:')
print(nulos_iniciales[nulos_iniciales > 0])
print()

# Imputación en catalog: nutriscore
if catalog['nutriscore'].isnull().sum() > 0:
    nulos_nutriscore = catalog['nutriscore'].isnull().sum()
    # Usar moda global si no hay suficientes por categoría
    moda_nutriscore = catalog['nutriscore'].mode()[0]
    catalog['nutriscore'] = catalog['nutriscore'].fillna(moda_nutriscore)
    print(f'Nutriscore: {nulos_nutriscore} nulos imputados con moda={moda_nutriscore}')

# Imputación en catalog: proteínas
if catalog['proteins_100g'].isnull().sum() > 0:
    nulos_proteinas = catalog['proteins_100g'].isnull().sum()
    media_proteinas = catalog['proteins_100g'].mean()
    catalog['proteins_100g'] = catalog['proteins_100g'].fillna(media_proteinas)
    print(f'Proteínas: {nulos_proteinas} nulos imputados con media={media_proteinas:.2f}')

# Imputación en catalog: carbohidratos
if catalog['carbs_100g'].isnull().sum() > 0:
    nulos_carbs = catalog['carbs_100g'].isnull().sum()
    media_carbs = catalog['carbs_100g'].mean()
    catalog['carbs_100g'] = catalog['carbs_100g'].fillna(media_carbs)
    print(f'Carbohidratos: {nulos_carbs} nulos imputados con media={media_carbs:.2f}')

# Imputación en catalog: calorías
if catalog['calories_100g'].isnull().sum() > 0:
    nulos_cal = catalog['calories_100g'].isnull().sum()
    media_cal = catalog['calories_100g'].mean()
    catalog['calories_100g'] = catalog['calories_100g'].fillna(media_cal)
    print(f'Calorías: {nulos_cal} nulos imputados con media={media_cal:.2f}')

# En movements: expiry_date es estructural (solo IN), se mantienen nulos
nulos_expiry = movements['expiry_date'].isnull().sum()
print(f'\nExpiry_date en movements: {nulos_expiry} nulos (esperado en OUT)')

log['transformaciones']['imputacion'] = {
    'nutriscore_imputados': nulos_nutriscore if 'nulos_nutriscore' in locals() else 0,
    'proteins_imputadas': nulos_proteinas if 'nulos_proteinas' in locals() else 0,
    'carbs_imputados': nulos_carbs if 'nulos_carbs' in locals() else 0,
    'calories_imputadas': nulos_cal if 'nulos_cal' in locals() else 0,
    'expiry_date_nulos_mantendidos': nulos_expiry
}

# Realizar JOIN: movements + catalog
inventory = movements.merge(
    catalog[['product_id', 'product_name', 'category_name', 
             'nutriscore', 'calories_100g', 'proteins_100g', 'carbs_100g']],
    on='product_id',
    how='left'
)

print(f'Inventory (después del JOIN): {inventory.shape[0]} filas × {inventory.shape[1]} columnas')
print(f'\nColumnas en inventory:')
print(list(inventory.columns))
print(f'\nPrimeros registros:')
print(inventory.head())

log['transformaciones']['join'] = {
    'tabla_izquierda': 'movements',
    'tabla_derecha': 'catalog',
    'tipo_join': 'left',
    'clave': 'product_id',
    'registros_resultado': len(inventory)
}

In [ ]:
# Crear columna derivada: Dias_Para_Vencer (solo para IN)
today = pd.Timestamp('today')
inventory['dias_para_vencer'] = inventory.apply(
    lambda row: (row['expiry_date'] - today).days if pd.notna(row['expiry_date']) else np.nan,
    axis=1
)

# Reordenar columnas para legibilidad
column_order = [
    'event_id', 'product_id', 'product_name', 'category_name',
    'timestamp', 'action_type', 'location', 
    'expiry_date', 'dias_para_vencer',
    'nutriscore', 'calories_100g', 'proteins_100g', 'carbs_100g'
]

inventory = inventory[column_order]

print('Estructura final de inventory_v1:')
print(inventory.dtypes)
print(f'\nNulos por columna (sample):')
print(inventory.isnull().sum())

log['transformaciones']['derivadas'] = {
    'dias_para_vencer_creada': True,
    'columnas_finales': len(inventory.columns)
}

## Paso 5: Validación y filtrado de outliers

In [ ]:
# Detectar outliers en calorías (umbral 900 kcal)
outliers_cal = catalog[catalog['calories_100g'] > 900]
print(f'Outliers detectados en calories_100g (>900): {len(outliers_cal)}')
if len(outliers_cal) > 0:
    print(outliers_cal[['product_id', 'product_name', 'calories_100g']])

# Remover outliers
registros_antes = len(catalog)
catalog = catalog[catalog['calories_100g'] <= 900]
outliers_removidos = registros_antes - len(catalog)

print(f'\nRegistros removidos: {outliers_removidos}')
print(f'Registros en catálogo después: {len(catalog)}')

log['transformaciones']['validacion_outliers'] = {
    'outliers_detectados': len(outliers_cal),
    'outliers_removidos': outliers_removidos,
    'umbral_calories_100g': 900,
    'registros_catalog_despues': len(catalog)
}

## Paso 6: Homologación de categorías

In [ ]:
# Mapeo de categorías numéricas a nombres estándar
# Basado en el muestreo de datos iniciales
category_mapping = {
    1: 'Beverages',
    2: 'Snacks',
    3: 'Dairy',
    4: 'Produce',
    5: 'Proteins',
    6: 'Grains',
    7: 'Condiments',
    8: 'Frozen Foods',
    9: 'Prepared Foods',
    10: 'Other'
}

# Aplicar mapeo (también crear columna numérica original)
catalog['category_id'] = catalog['category']
catalog['category_name'] = catalog['category'].map(category_mapping).fillna('Unknown')

print('Distribución de categorías:')
print(catalog['category_name'].value_counts())

log['transformaciones']['homologacion_categorias'] = {
    'mapeo_aplicado': True,
    'categorias_distintas': catalog['category_name'].nunique()
}

## Paso 7: Validación de integridad referencial

In [ ]:
# Verificar que todos los product_id en movements existen en catalog
huerfanos = movements[~movements['product_id'].isin(catalog['product_id'])]
print(f'Eventos con product_id ausente en catálogo: {len(huerfanos)}')

if len(huerfanos) > 0:
    print('\nMuestreo de eventos huérfanos:')
    print(huerfanos[['event_id', 'product_id']].head())
    # Remover eventos huérfanos
    movements = movements[movements['product_id'].isin(catalog['product_id'])]
    print(f'\nRegistros en movements después de remover huérfanos: {len(movements)}')

log['transformaciones']['integridad_referencial'] = {
    'huerfanos_detectados': len(huerfanos),
    'huerfanos_removidos': len(huerfanos),
    'registros_movements_despues': len(movements)
}

## Paso 8: JOIN de tablas

In [ ]:
# Realizar JOIN: movements + catalog
inventory = movements.merge(
    catalog[['product_id', 'product_name', 'category_id', 'category_name', 
             'nutriscore', 'calories_100g', 'proteins_100g', 'carbs_100g']],
    on='product_id',
    how='left'
)

print(f'Inventory (después del JOIN): {inventory.shape[0]} filas × {inventory.shape[1]} columnas')
print(f'\nColumnas en inventory:')
print(list(inventory.columns))
print(f'\nPrimeros registros:')
print(inventory.head())

log['transformaciones']['join'] = {
    'tabla_izquierda': 'movements',
    'tabla_derecha': 'catalog',
    'tipo_join': 'left',
    'clave': 'product_id',
    'registros_resultado': len(inventory)
}

## Paso 9: Derivadas y validación final

In [ ]:
# Crear columna derivada: Dias_Para_Vencer (solo para IN)
today = pd.Timestamp('today')
inventory['dias_para_vencer'] = inventory.apply(
    lambda row: (row['expiry_date'] - today).days if pd.notna(row['expiry_date']) else np.nan,
    axis=1
)

# Reordenar columnas para legibilidad
column_order = [
    'event_id', 'product_id', 'product_name', 'category_id', 'category_name',
    'timestamp', 'action_type', 'location', 'quantity', 
    'expiry_date', 'dias_para_vencer',
    'nutriscore', 'calories_100g', 'proteins_100g', 'carbs_100g'
]

inventory = inventory[column_order]

print('Estructura final de inventory_v1:')
print(inventory.dtypes)
print(f'\nNulos por columna (sample):')
print(inventory.isnull().sum())

log['transformaciones']['derivadas'] = {
    'dias_para_vencer_creada': True,
    'columnas_finales': len(inventory.columns)
}

## Paso 10: Exportación y bitácora

In [ ]:
# Guardar inventory_v1.csv
output_path = '../data/interim/inventory_v1.csv'
inventory.to_csv(output_path, index=False)
print(f'✓ Archivo guardado: {output_path}')
print(f'  Tamaño: {len(inventory)} registros')

# Completar bitácora
log['registros_finales'] = {
    'catalog': len(catalog),
    'movements': len(movements),
    'inventory_v1': len(inventory)
}

log['calidad_datos'] = {
    'nulos_por_columna': inventory.isnull().sum().to_dict(),
    'cardinalidad_category_name': int(inventory['category_name'].nunique()),
    'rango_temporal': {
        'min': str(inventory['timestamp'].min()),
        'max': str(inventory['timestamp'].max())
    }
}

# Guardar bitácora
log_path = '../data/interim/transformations_log.json'
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2, default=str)
print(f'✓ Bitácora guardada: {log_path}')

# Resumen final
print(f'\n=== RESUMEN DE TRANSFORMACIÓN ===')
print(f'Registros iniciales (movements): {log["registros_iniciales"]["movements"]}')
print(f'Registros finales (inventory_v1): {log["registros_finales"]["inventory_v1"]}')
print(f'Registros removidos: {log["registros_iniciales"]["movements"] - log["registros_finales"]["inventory_v1"]}')
print(f'Productos en catálogo final: {log["registros_finales"]["catalog"]}')

## Verificación final

In [ ]:
# Verificación: cargar archivo generado
inventory_check = pd.read_csv('../data/interim/inventory_v1.csv')
print(f'Archivo generado verificado: {len(inventory_check)} registros')
print(f'\nMuestras de categorías:')
print(inventory_check['category_name'].value_counts())
print(f'\nTipos finales:')
print(inventory_check.dtypes)